In [ ]:
import polars as pl

EVENT_ID = 597

In [ ]:
df_relay = pl.read_csv(f"{EVENT_ID}-riepilogo-staffette-dbmeeting.csv", separator=";")

# Identify columns related to 'Atleta'
atleta_cols = [col for col in df_relay.columns if col.startswith("Atleta")]

# Combine into a single list column, removing duplicates and nulls
df_relay = df_relay.with_columns(
    pl.concat_list(atleta_cols)
    .list.drop_nulls()
    .list.unique(maintain_order=True)
    .alias("Atleta")
).drop([c for c in atleta_cols if c != "Atleta"])

df_relay.head()

In [ ]:
df_individual = pl.read_csv(f"{EVENT_ID}-riepilogo-dbmeeting.csv", separator=";")

In [ ]:
# Create a mapping of Athlete Name -> Birth Year
df_ages = df_individual.select(
    pl.concat_str([pl.col("Cognome"), pl.col("Nome")], separator=" ").alias("Atleta"),
    pl.col("Anno")
).unique()

# Add a row index to preserve order and group back later
df_relay = df_relay.with_row_index("row_nr")

# Explode, Join, Calculate Age, GroupBy, Sum
df_relay_ages = (
    df_relay.explode("Atleta")
    .join(df_ages, on="Atleta", how="left")
    .with_columns((2026 - pl.col("Anno")).alias("Age"))
    .group_by("row_nr")
    .agg(pl.col("Age").sum().alias("SumAges"))
)

# Join back to the original dataframe
df_relay = df_relay.join(df_relay_ages, on="row_nr").drop("row_nr")

df_relay.head()

In [ ]:
# Define function to determine category
def get_relay_category(age_sum):
    if age_sum < 60:
        return "FG"
    return chr(ord('A') + (age_sum - 60) // 30)

# Apply function and reorder columns
df_relay = df_relay.with_columns(
    pl.col("SumAges").map_elements(get_relay_category, return_dtype=pl.String).alias("CategoriaStaffetta")
).select(["CategoriaStaffetta"] + [c for c in df_relay.columns if c != "CategoriaStaffetta"])

df_relay.head()

In [ ]:
# Check max number of athletes
max_athletes = df_relay["Atleta"].list.len().max()
print(f"Max unique athletes per row: {max_athletes}")

# Split 'Atleta' list into separate columns safely
# User requested 3 columns
for i in range(3):
    df_relay = df_relay.with_columns(
        pl.col("Atleta").list.slice(i, 1).list.first().alias(f"Atleta_{i}")
    )

# Drop unnecessary columns
df_final = df_relay.drop(["Atleta", "SumAges"])

# Rename columns to match requirement (we can't have duplicate column names in DataFrame, so we handle it at write time)
print(df_final.head())

# Save to CSV with custom header to have multiple "Atleta" columns
output_filename = f"{EVENT_ID}-staffette-processed.csv"

# We construct the header manually
header = df_final.columns
# Replace Atleta_0, Atleta_1, Atleta_2 with Atleta
custom_header = [c if not c.startswith("Atleta_") else "Atleta" for c in header]
custom_header_str = ";".join(custom_header)

# Write body using polars (without header)
df_final.write_csv(output_filename, separator=";", include_header=False)

# Prepend the custom header
with open(output_filename, 'r+') as f:
    content = f.read()
    f.seek(0, 0)
    f.write(custom_header_str + '\n' + content)

print(f"Saved to {output_filename}")